

##  Model 4 — Targeted Fixes from Error Analysis

This iteration introduces **three surgical feature engineering changes** designed to directly solve the weaknesses exposed in the baseline validation analysis. All underlying model hyperparameters, architectural structures, and the recursive inference engine remain identical to v2/v3.

---

### 📈 FIX 1 — Dynamic Rolling YoY Ratio *(Cell 3 Addition)*

* **Root Cause:** Stores 2, 7, 8, and 9 experienced a massive 20% to 50% structural revenue surge in summer 2015 compared to historical baselines. Because the baseline rolling window features (`rmean_28`, `rmean_56`) were heavily influenced by 2011–2014 trailing data, they kept pulling the recursive predictions back down toward the lower historical average.
* **Solution:** Introduced a dynamic, un-smoothed tracking feature:

$$\text{rolling\_yoy\_ratio} = \frac{\text{recent\_4wk\_revenue}}{\text{same\_4wk\_revenue\_last\_year}}$$



This is computed dynamically per-store and per-day. It explicitly flags to LightGBM when a store is currently operating at an accelerated volume relative to its 12-month baseline. During the recursive inference loop, this ratio updates automatically as predictions accumulate.

---

### 📅 FIX 2 — Independence Day × DoW Interaction *(Cell 2 + Cell 3)*

* **Root Cause:** The baseline model suffered a massive error ($\text{MAE} = 8,075$) on Independence Day. Residual analysis proved the holiday's impact is entirely conditional on the day of the week it falls on (e.g., Monday = $-37\%$, Wednesday = $-2\%$, Friday = $+9\%$, Saturday = $+12\%$). Passing `IndependenceDay` as a simple, static categorical variable left the tree blind to this scheduling drift.
* **Solution:** Added two explicit temporal interaction features:
* `jul4_dow`: Evaluates to the integer `day_of_week` when `is_jul4 == 1`, otherwise defaults to `-1`.
* `is_jul3`: A dedicated binary flag to isolate the highly compressed, pre-holiday grocery and retail surge (consistently tracking at $+15\%$ to $+33\%$).



---

### ⚡ FIX 3 — Multiplicative Event-Lag Interaction *(Cell 3)*

* **Root Cause:** Global calendar event flags consistently ranked below the top 15 in information gain feature importance. Because lag features carry massive numerical variance (e.g., store revenue fluctuating from 20,000 to 45,000), the tree branches prioritized splitting on those high-variance values, effectively drowning out the low-variance $0$ or $1$ binary event triggers.
* **Solution:** Re-framed the interaction mathematically by calculating a multiplicative constraint directly:

$$\text{lag7\_x\_event} = \text{lag\_7} \times (1 + \text{event\_lift\_value})$$



By forcing the event signal to directly scale the baseline volume trajectory instead of competing additively against it, the tree is forced to prioritize the holiday's relative impact during information split decisions.

---

### 🔒 Structural Baseline (Not Changed)

To ensure an isolated and controlled experiment, the following pipeline blocks have **not** been altered:

* Global and store-level model hyperparameters.
* The fundamental rolling window and cross-store lag structure.
* The day-by-day sequential recursive prediction loop.

In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import mean_squared_error
import pickle, os, warnings
warnings.filterwarnings('ignore')

# ── 0. LOAD ────────────────────────────────────────────────────────────────────
train    = pd.read_csv('data/train.csv')
cal      = pd.read_csv('data/calendar_events.csv')
sub_tmpl = pd.read_csv('data/forecast_submission.csv')
for df_ in [train, cal, sub_tmpl]:
    df_.columns = df_.columns.str.strip().str.lower()
train['date'] = pd.to_datetime(train['date'])
cal['date']   = pd.to_datetime(cal['date'])
train = train[train['store_id'] > 0].copy()

# ── 1. CALENDAR SETUP (Cell 2 — unchanged except jul3 addition) ───────────────

In [2]:

NEGATIVE_EVENTS = {
    'Christmas','Thanksgiving','NewYear',"Mother's day",'Halloween',
    "Father's day",'Easter','OrthodoxEaster, Easter','ValentinesDay',
    'Easter, OrthodoxEaster','OrthodoxEaster, Cinco De Mayo',
    'IndependenceDay','SuperBowl','StPatricksDay','LentWeek2',
    'LentStart','Pesach End','NBAFinalsEnd',
}
POSITIVE_EVENTS = {
    'LaborDay','OrthodoxEaster','NBAFinalsStart','EidAlAdha','MemorialDay',
    'Ramadan starts',"NBAFinalsEnd, Father's day",'VeteransDay','ColumbusDay',
    'MartinLutherKingDay','Purim End','Chanukah End','Eid al-Fitr',
    'PreIndependenceDay',
}
EVENT_LIFT = {
    'LaborDay':0.291,'OrthodoxEaster':0.132,'NBAFinalsStart':0.067,
    'EidAlAdha':0.057,'MemorialDay':0.044,'Ramadan starts':0.044,
    "NBAFinalsEnd, Father's day":0.055,'VeteransDay':0.031,
    'ColumbusDay':0.030,'MartinLutherKingDay':0.019,'PreIndependenceDay':0.25,
    'Chanukah End':0.014,'Eid al-Fitr':0.017,'Purim End':0.030,
    'Christmas':-1.0,'Thanksgiving':-0.337,'NewYear':-0.232,
    "Mother's day":-0.150,'Halloween':-0.129,"Father's day":-0.128,
    'Easter':-0.068,'ValentinesDay':-0.077,
    # FIX 2: IndependenceDay lift is DoW-dependent — set base to 0
    # the jul4_dow feature captures the actual effect
    'IndependenceDay': 0.0,
    'SuperBowl':-0.037,'OrthodoxEaster, Easter':-0.424,
}

# Augment calendar: add PreIndependenceDay (Jul 3) each year
extra = [{'date': pd.Timestamp(f'{y}-07-03'), 'event': 'PreIndependenceDay'}
         for y in range(2011, 2018)
         if pd.Timestamp(f'{y}-07-03') not in set(cal['date'])]
cal_aug    = pd.concat([cal, pd.DataFrame(extra)], ignore_index=True)
cal_aug['date'] = pd.to_datetime(cal_aug['date'])
cal_lookup = dict(zip(cal_aug['date'], cal_aug['event']))
cal_sorted = sorted(set(cal_aug['date']))

def nearest_offset(d):
    diffs = [(d - e).days for e in cal_sorted]
    return int(np.clip(diffs[int(np.argmin(np.abs(diffs)))], -21, 21))

def hev(d):
    return int(cal_lookup.get(d, 'NoEvent') != 'NoEvent')

# ── 2. YoY STATIC RATIOS (unchanged from v3) ──────────────────────────────────

In [3]:
yoy_store = {}
yoy_store_month = {}
tmp = train.copy()
tmp['year']  = tmp['date'].dt.year
tmp['month'] = tmp['date'].dt.month
for sid in sorted(train['store_id'].unique()):
    s = tmp[tmp['store_id']==sid]
    v14 = s[(s['year']==2014)&(s['month']<=9)]['revenue'].mean()
    v15 = s[(s['year']==2015)&(s['month']<=9)]['revenue'].mean()
    yoy_store[sid] = v15 / v14
    for m in range(1, 13):
        v14m = s[(s['year']==2014)&(s['month']==m)]['revenue'].mean()
        v15m = s[(s['year']==2015)&(s['month']==m)]['revenue'].mean()
        yoy_store_month[(sid, m)] = (v15m/v14m) if (not np.isnan(v14m) and not np.isnan(v15m)) else yoy_store[sid]

# ── 3. FEATURE ENGINEERING (Cell 3 — three additions marked FIX 1/2/3) ────────

In [10]:
def build_features(df):
    df = df.sort_values(['store_id', 'date']).reset_index(drop=True)

    # ── Time Features ──
    df['dow']            = df['date'].dt.dayofweek
    df['day_of_month']   = df['date'].dt.day
    df['month']          = df['date'].dt.month
    df['year']           = df['date'].dt.year
    df['week_of_year']   = df['date'].dt.isocalendar().week.astype(int)
    df['quarter']        = df['date'].dt.quarter
    df['is_weekend']     = df['dow'].isin([5, 6]).astype(int)
    df['is_month_start'] = df['date'].dt.is_month_start.astype(int)
    df['is_month_end']   = df['date'].dt.is_month_end.astype(int)
    df['days_since_start'] = (df['date'] - pd.Timestamp('2011-01-29')).dt.days
    df['dow_sin']   = np.sin(2*np.pi*df['dow']/7)
    df['dow_cos']   = np.cos(2*np.pi*df['dow']/7)
    df['month_sin'] = np.sin(2*np.pi*df['month']/12)
    df['month_cos'] = np.cos(2*np.pi*df['month']/12)

    # ── Categorical Interactions ──
    df['store_dow']     = df['store_id'].astype(str) + '_' + df['dow'].astype(str)
    df['store_weekend'] = df['store_id'].astype(str) + '_' + df['is_weekend'].astype(str)
    df['store_month']   = df['store_id'].astype(str) + '_' + df['month'].astype(str)

    # ── Events Base Mapping ──
    df['event']           = df['date'].map(cal_lookup).fillna('NoEvent')
    df['event_negative']  = df['event'].isin(NEGATIVE_EVENTS).astype(int)
    df['event_positive']  = df['event'].isin(POSITIVE_EVENTS).astype(int)
    df['event_lift_val']  = df['event'].map(EVENT_LIFT).fillna(0.0)
    df['days_from_event'] = df['date'].apply(nearest_offset)

    # ── FIX 2: Independence Day × DoW interaction ──
    df['is_jul4']  = ((df['month'] == 7) & (df['day_of_month'] == 4)).astype(int)
    df['jul4_dow'] = np.where(df['is_jul4'] == 1, df['dow'], -1)
    df['is_jul3']  = ((df['month'] == 7) & (df['day_of_month'] == 3)).astype(int)

    return df


def add_lags_and_rolling(df):
    df = df.sort_values(['store_id', 'date']).reset_index(drop=True)
    grp = df.groupby('store_id')['revenue']

    # Lags (unchanged)
    for lag in [1, 2, 3, 6, 7, 14, 21, 28, 35, 42, 56]:
        df[f'lag_{lag}'] = grp.shift(lag)

    # Annual lags with static YoY scaling (unchanged)
    for lag in [364, 365, 371]:
        df[f'lag_{lag}'] = grp.shift(lag)
    df['store_yoy']         = df['store_id'].map(yoy_store)
    df['store_yoy_monthly'] = df.apply(
        lambda r: yoy_store_month.get((r['store_id'], r['month']), yoy_store[r['store_id']]), axis=1)
    df['lag_365_scaled'] = df['lag_365'] * df['store_yoy_monthly']
    df['lag_371_scaled'] = df['lag_371'] * df['store_yoy_monthly']

    # Rolling stats (unchanged)
    s7 = grp.shift(7)
    for w in [7, 14, 28, 56]:
        df[f'rmean_{w}'] = s7.groupby(df['store_id']).transform(
            lambda x: x.rolling(w, min_periods=1).mean())
        df[f'rstd_{w}']  = s7.groupby(df['store_id']).transform(
            lambda x: x.rolling(w, min_periods=1).std().fillna(0))
        df[f'rmax_{w}']  = s7.groupby(df['store_id']).transform(
            lambda x: x.rolling(w, min_periods=1).max())
        df[f'rmin_{w}']  = s7.groupby(df['store_id']).transform(
            lambda x: x.rolling(w, min_periods=1).min())

    df['lag7_mean_4w']  = (df['lag_7']+df['lag_14']+df['lag_21']+df['lag_28'])/4
    df['lag7_mean_8w']  = df[[f'lag_{d}' for d in [7,14,21,28,35,42]]].mean(axis=1)
    df['momentum_7_28'] = df['lag_7'] / (df['lag_28'] + 1e-6)

    # ── FIX 1: Dynamic rolling YoY ratio ──────────────────────────────────────
    # recent_4wk_mean / same_4wk_mean_last_year
    recent_4wk = s7.groupby(df['store_id']).transform(
        lambda x: x.rolling(28, min_periods=14).mean())
    same_4wk_ly = grp.shift(365).groupby(df['store_id']).transform(
        lambda x: x.rolling(28, min_periods=14).mean())
    df['rolling_yoy_ratio'] = (recent_4wk / (same_4wk_ly + 1e-6)).clip(0.5, 3.0)
    
    # Recent 4wk vs train-global mean per store
    store_global_mean = df.groupby('store_id')['revenue'].transform('mean')
    df['rev_level_ratio'] = recent_4wk / (store_global_mean + 1e-6)

    # ── FIX 3: Multiplicative event-lag interaction ────────────────────────────
    df['lag7_x_event']  = df['lag_7']  * (1 + df['event_lift_val'])
    df['lag28_x_event'] = df['lag_28'] * (1 + df['event_lift_val'])
    
    # ── CRITICAL FIX: Safe Event Key Fallbacks ──
    # If the parent dataframe has the pre-computed keys, use them. 
    # If the prediction loop passes a single row missing them, map them safely from cal_lookup.
    if 'event_tomorrow' in df.columns:
        df['lag7_x_ev_tmrw'] = df['lag_7'] * df['event_tomorrow']
    else:
        # Look forward 1 day using the current date index map
        ev_tmrw_flag = df['date'].apply(lambda d: int(cal_lookup.get(d + pd.Timedelta(days=1), 'NoEvent') != 'NoEvent'))
        df['lag7_x_ev_tmrw'] = df['lag_7'] * ev_tmrw_flag

    if 'event_yesterday' in df.columns:
        df['lag7_x_ev_yest'] = df['lag_7'] * df['event_yesterday']
    else:
        # Look backward 1 day using the current date index map
        ev_yest_flag = df['date'].apply(lambda d: int(cal_lookup.get(d - pd.Timedelta(days=1), 'NoEvent') != 'NoEvent'))
        df['lag7_x_ev_yest'] = df['lag_7'] * ev_yest_flag

    return df


def add_cross_store(df):
    """Unchanged from v2."""
    pivot = df.pivot_table(index='date', columns='store_id', values='lag_1', aggfunc='first')
    pairs = [(8,4),(8,7),(8,3),(9,4),(3,4),(1,3),(9,3),(7,4)]
    for src, tgt in pairs:
        if src in pivot.columns:
            col = f'xstore_{src}_for_{tgt}'
            src_lag1 = pivot[src]
            df[col] = df.apply(
                lambda r: src_lag1.get(r['date'], np.nan) if r['store_id'] == tgt else np.nan,
                axis=1
            )
    return df


print("Building features...")

# 1. Run point-in-time features (Fix 2 is handled here)
df = build_features(train.copy())

# 2. Run lag and rolling statistics function (Fix 1 and baseline stats are handled here)
df = add_lags_and_rolling(df)

# 3. ── CRITICAL ADDITION: Pre-compute global holiday shift windows ──
# This must happen before cross-store calculations so all variables are locked down
df = df.sort_values(['store_id', 'date']).reset_index(drop=True)
ev_bin = (df['event'] != 'NoEvent').astype(float)
df['is_event_temp'] = ev_bin

print("Pre-computing global holiday shift windows...")
for shift, name in [(-1, 'event_tomorrow'), (-2, 'event_in_2d'), 
                    (1, 'event_yesterday'), (2, 'event_2d_ago'), (7, 'event_week_ago')]:
    df[name] = df.groupby('store_id')['is_event_temp'].transform(lambda x: x.shift(shift)).fillna(0).astype(int)

# Drop the temporary calculation helper column
df = df.drop(columns=['is_event_temp'])

# 4. Set up cross-store external variables (unchanged structure)
df = add_cross_store(df)

# 5. Clean up boundary missing data based on your specific lag floor
df = df.dropna(subset=['lag_42']).reset_index(drop=True)

print(f"Feature df shape fully prepared: {df.shape}")

Building features...
Pre-computing global holiday shift windows...
Feature df shape fully prepared: (16640, 85)


# ── 4. FEATURE LIST ────────────────────────────────────────────────────────────

In [11]:
CAT_FEATURES = ['store_id', 'event', 'store_weekend', 'store_dow', 'store_month']
TIME_FEATURES = [
    'dow','day_of_month','month','year','week_of_year','quarter',
    'is_weekend','is_month_start','is_month_end','days_since_start',
    'dow_sin','dow_cos','month_sin','month_cos',
]
LAG_FEATURES = (
    [f'lag_{d}' for d in [1,2,3,6,7,14,21,28,35,42,56]] +
    ['lag_364','lag_365','lag_371','lag_365_scaled','lag_371_scaled',
     'lag7_mean_4w','lag7_mean_8w']
)
ROLLING_FEATURES = (
    [f'{s}_{w}' for s in ['rmean','rstd','rmax','rmin'] for w in [7,14,28,56]] +
    ['momentum_7_28']
)
EVENT_FEATURES = [
    'event_negative','event_positive','event_lift_val','days_from_event',
    'event_in_2d','event_tomorrow','event_yesterday','event_2d_ago','event_week_ago',
]
# NEW FEATURES from fixes
FIX_FEATURES = [
    # FIX 1
    'rolling_yoy_ratio', 'rev_level_ratio', 'store_yoy', 'store_yoy_monthly',
    # FIX 2
    'is_jul4', 'jul4_dow', 'is_jul3',
    # FIX 3
    'lag7_x_event', 'lag28_x_event', 'lag7_x_ev_tmrw', 'lag7_x_ev_yest',
]
XSTORE_FEATURES = [c for c in df.columns if c.startswith('xstore_')]

FEATURES = (CAT_FEATURES + TIME_FEATURES + LAG_FEATURES +
            ROLLING_FEATURES + EVENT_FEATURES + FIX_FEATURES + XSTORE_FEATURES)
TARGET = 'revenue'

for col in CAT_FEATURES:
    df[col] = df[col].astype('category')

print(f"Total features: {len(FEATURES)}  (added {len(FIX_FEATURES)} new)")

Total features: 82  (added 11 new)


# ── 5. TRAIN / VALIDATION SPLIT ───────────────────────────────────────────────

In [13]:
VALID_DAYS = 120
val_start = pd.Timestamp('2015-01-01')
val_end   = pd.Timestamp('2015-03-31')

# Train and Validation seasonal split chunks
train_df  = df[(df['date'] < val_start) | (df['date'] > val_end)]
valid_df  = df[(df['date'] >= val_start) & (df['date'] <= val_end)]

X_train, y_train = train_df[FEATURES], train_df[TARGET]
X_valid, y_valid = valid_df[FEATURES],  valid_df[TARGET]
print(f"Train rows: {len(X_train):,}  |  Valid rows: {len(X_valid):,}")


Train rows: 15,740  |  Valid rows: 900


# ── 6. TRAIN ──────────────────────────────────────────────────────────────────

In [14]:
model = lgb.LGBMRegressor(
    objective         = 'regression_l2',
    n_estimators      = 5000,
    learning_rate     = 0.02,
    num_leaves        = 255,
    max_depth         = -1,
    min_child_samples = 15,
    feature_fraction  = 0.75,
    bagging_fraction  = 0.75,
    bagging_freq      = 1,
    reg_alpha         = 0.05,
    reg_lambda        = 0.1,
    random_state      = 42,
    n_jobs            = -1,
    verbose           = -1,
)
print("Training...")
model.fit(
    X_train, y_train,
    eval_set  = [(X_valid, y_valid)],
    callbacks = [
        lgb.early_stopping(stopping_rounds=150, verbose=True),
        lgb.log_evaluation(250),
    ],
    categorical_feature = CAT_FEATURES,
)
valid_preds = model.predict(X_valid)
rmse = np.sqrt(mean_squared_error(y_valid, valid_preds))
print(f"\n✅ Validation RMSE: {rmse:.2f}")

# Per-store RMSE + bias (compare directly to error analysis output)
print("\nPer-store breakdown:")
print(f"{'Store':<8} {'RMSE':>8} {'Bias':>8} {'MAPE':>8}")
for sid in sorted(valid_df['store_id'].unique()):
    sv = valid_df[valid_df['store_id']==sid]
    sp = model.predict(sv[FEATURES])
    sr = np.sqrt(mean_squared_error(sv[TARGET], sp))
    sb = (sv[TARGET].values - sp).mean()
    sm = (np.abs(sv[TARGET].values - sp) / sv[TARGET].values.clip(1) * 100).mean()
    print(f"  {sid:<6} {sr:>8,.0f} {sb:>+8,.0f} {sm:>7.1f}%")

# Feature importance — check if new features rank highly
fi = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False)
print("\nTop 25 features:")
print(fi.head(25).to_string())
print("\nNew FIX features ranking:")
for f in FIX_FEATURES:
    rank = list(fi.index).index(f) + 1
    print(f"  {f:<30} rank {rank:>3}  importance {fi[f]:,.0f}")

os.makedirs('data', exist_ok=True)
with open('data/model_v4.pkl', 'wb') as f_: pickle.dump(model, f_)
valid_df_out = valid_df.copy()
valid_df_out['pred'] = valid_preds
valid_df_out.to_csv('data/valid_predictions_v4.csv', index=False)
fi.to_csv('data/feature_importance_v4.csv', header=['importance'])
print("\nSaved model_v4.pkl, valid_predictions_v4.csv, feature_importance_v4.csv")

Training...
Training until validation scores don't improve for 150 rounds
[250]	valid_0's l2: 7.62111e+06
[500]	valid_0's l2: 7.47919e+06
[750]	valid_0's l2: 7.47094e+06
Early stopping, best iteration is:
[728]	valid_0's l2: 7.46764e+06

✅ Validation RMSE: 2732.70

Per-store breakdown:
Store        RMSE     Bias     MAPE
  1         2,150      -49     5.4%
  2         1,768     -737     7.3%
  3         3,195     +558     5.0%
  4         1,199     +168     5.2%
  5         1,653     +385     5.6%
  6         4,263     +592    30.1%
  7         2,290     +415     6.8%
  8         3,541       -7     9.3%
  9         3,171     -281     8.1%
  10        2,558      +20     8.5%

Top 25 features:
day_of_month         8106
momentum_7_28        6791
lag_1                6778
lag_364              6211
lag_6                5681
rolling_yoy_ratio    5566
lag_56               5018
store_month          4912
lag_3                4802
rstd_7               4694
lag_2                4672
lag_365_scale

# ── 7. RECURSIVE INFERENCE ────────────────────────────────────────────────────

In [ ]:
print("\n── Recursive Inference ──")
sub = sub_tmpl.copy()
sub['store_id'] = sub['id'].apply(lambda x: int(x.split('_')[0]))
sub['date']     = pd.to_datetime(sub['id'].apply(lambda x: x.split('_')[1]), format='%Y%m%d')
sub = sub.sort_values(['store_id', 'date']).reset_index(drop=True)

store_ids      = sorted(sub['store_id'].unique())
forecast_start = sub['date'].min()
forecast_end   = sub['date'].max()
global_min     = pd.Timestamp('2011-01-29')

history = {}
for sid, grp in train.groupby('store_id'):
    history[int(sid)] = dict(zip(pd.to_datetime(grp['date']), grp['revenue'].values))

def get_lag(sid, date, lag_days):
    return history[sid].get(date - pd.Timedelta(days=lag_days), np.nan)

def get_window_vals(sid, date, shift, window):
    base = date - pd.Timedelta(days=shift)
    return [history[sid][base - pd.Timedelta(days=i)]
            for i in range(window)
            if (base - pd.Timedelta(days=i)) in history[sid]]

results = []
for date in pd.date_range(forecast_start, forecast_end, freq='D'):
    rows = []
    for sid in store_ids:
        ev = cal_lookup.get(date, 'NoEvent')
        ev_lift = EVENT_LIFT.get(ev, 0.0)
        yoy_r   = yoy_store[sid]
        yoy_m   = yoy_store_month.get((sid, date.month), yoy_r)

        # Lags
        lv = {f'lag_{d}': get_lag(sid, date, d)
              for d in [1,2,3,6,7,14,21,28,35,42,56,364,365,371]}
        l7  = lv.get('lag_7',  np.nan)
        l28 = lv.get('lag_28', np.nan)
        l365= lv.get('lag_365',np.nan)

        # Rolling
        rol = {}
        for w in [7, 14, 28, 56]:
            vals = get_window_vals(sid, date, 7, w)
            rol[f'rmean_{w}'] = float(np.mean(vals)) if vals else np.nan
            rol[f'rstd_{w}']  = float(np.std(vals))  if len(vals)>1 else 0.0
            rol[f'rmax_{w}']  = float(np.max(vals))  if vals else np.nan
            rol[f'rmin_{w}']  = float(np.min(vals))  if vals else np.nan
        fb = rol.get('rmean_28', rol.get('rmean_14', l7 or 0))

        # FIX 1: Dynamic rolling YoY ratio
        recent_vals  = get_window_vals(sid, date, 7, 28)
        ly_vals      = get_window_vals(sid, date, 7+365, 28)
        recent_mean  = float(np.mean(recent_vals)) if recent_vals else fb
        ly_mean      = float(np.mean(ly_vals))     if ly_vals     else (fb / yoy_m + 1e-6)
        rolling_yoy  = float(np.clip(recent_mean / (ly_mean + 1e-6), 0.5, 3.0))
        # Global store mean (approximate from training data)
        store_hist_vals = list(history[sid].values())
        store_global_m  = float(np.mean(store_hist_vals[-730:])) if len(store_hist_vals) >= 730 else float(np.mean(store_hist_vals))
        rev_level_ratio = recent_mean / (store_global_m + 1e-6)

        lv4 = np.nanmean([lv.get(f'lag_{d}', np.nan) for d in [7,14,21,28]])
        lv8 = np.nanmean([lv.get(f'lag_{d}', np.nan) for d in [7,14,21,28,35,42]])

        row = dict(
            store_id         = sid,
            event            = ev,
            store_weekend    = f'{sid}_{int(date.dayofweek in [5,6])}',
            store_dow        = f'{sid}_{date.dayofweek}',
            store_month      = f'{sid}_{date.month}',
            dow              = date.dayofweek,
            day_of_month     = date.day,
            month            = date.month,
            year             = date.year,
            week_of_year     = date.isocalendar()[1],
            quarter          = (date.month-1)//3+1,
            is_weekend       = int(date.dayofweek in [5,6]),
            is_month_start   = int(date.day==1),
            is_month_end     = int((date+pd.Timedelta(days=1)).month != date.month),
            days_since_start = (date - global_min).days,
            dow_sin          = np.sin(2*np.pi*date.dayofweek/7),
            dow_cos          = np.cos(2*np.pi*date.dayofweek/7),
            month_sin        = np.sin(2*np.pi*date.month/12),
            month_cos        = np.cos(2*np.pi*date.month/12),
            event_negative   = int(ev in NEGATIVE_EVENTS),
            event_positive   = int(ev in POSITIVE_EVENTS),
            event_lift_val   = ev_lift,
            days_from_event  = nearest_offset(date),
            event_in_2d      = hev(date+pd.Timedelta(days=2)),
            event_tomorrow   = hev(date+pd.Timedelta(days=1)),
            event_yesterday  = hev(date-pd.Timedelta(days=1)),
            event_2d_ago     = hev(date-pd.Timedelta(days=2)),
            event_week_ago   = hev(date-pd.Timedelta(days=7)),
            lag_365_scaled   = l365*yoy_m if not np.isnan(l365) else fb,
            lag_371_scaled   = (lv.get('lag_371',np.nan)*yoy_m) if not np.isnan(lv.get('lag_371',np.nan)) else fb,
            lag7_mean_4w     = lv4 if not np.isnan(lv4) else fb,
            lag7_mean_8w     = lv8 if not np.isnan(lv8) else fb,
            momentum_7_28    = l7/(l28+1e-6) if not np.isnan(l7) and not np.isnan(l28) else 1.0,
            store_yoy        = yoy_r,
            store_yoy_monthly= yoy_m,
            # FIX 1
            rolling_yoy_ratio= rolling_yoy,
            rev_level_ratio  = rev_level_ratio,
            # FIX 2
            is_jul4          = int(date.month==7 and date.day==4),
            jul4_dow         = date.dayofweek if (date.month==7 and date.day==4) else -1,
            is_jul3          = int(date.month==7 and date.day==3),
            # FIX 3
            lag7_x_event     = l7*(1+ev_lift)   if not np.isnan(l7)  else fb*(1+ev_lift),
            lag28_x_event    = l28*(1+ev_lift)  if not np.isnan(l28) else fb*(1+ev_lift),
            lag7_x_ev_tmrw   = (l7 if not np.isnan(l7) else fb) * hev(date+pd.Timedelta(days=1)),
            lag7_x_ev_yest   = (l7 if not np.isnan(l7) else fb) * hev(date-pd.Timedelta(days=1)),
            **lv, **rol,
        )

        # Cross-store
        for xf in XSTORE_FEATURES:
            parts = xf.replace('xstore_','').split('_for_')
            src, tgt = int(parts[0]), int(parts[1])
            row[xf] = history.get(src,{}).get(date-pd.Timedelta(days=1), np.nan) if sid==tgt else np.nan

        rows.append(row)

    X_day = pd.DataFrame(rows)[FEATURES].copy()
    for c in CAT_FEATURES:
        X_day[c] = X_day[c].astype('category')
    num_cols = X_day.select_dtypes(include=[np.number]).columns
    X_day[num_cols] = X_day[num_cols].fillna(
        X_day[['rmean_28','rmean_14','rmean_7']].mean(axis=1).values.reshape(-1,1)
    )

    preds = model.predict(X_day).clip(0)
    if cal_lookup.get(date,'NoEvent') == 'Christmas':
        preds *= 0.001

    for sid, pred in zip(store_ids, preds):
        history[sid][date] = float(pred)
        results.append({'id': f'{sid}_{date.strftime("%Y%m%d")}', 'prediction': float(pred)})

    if date.day == 1:
        print(f"  Forecasted through {date.strftime('%Y-%m-%d')}")

# ── 8. SAVE ────────────────────────────────────────────────────────────────────

In [ ]:

results_df = pd.DataFrame(results)
final = sub_tmpl[['id']].merge(results_df, on='id', how='left')
final['prediction'] = final['prediction'].fillna(final['prediction'].mean())
final.to_csv('submission_v4.csv', index=False)
print(f"\n✅ Saved 'submission_5.csv'  ({len(final)} rows)")
print(final.head(10).to_string(index=False))
print(f"\n{final['prediction'].describe()}")
